# Build Process Trees from DMBD dataset

Download dmbd data from https://gdo168.llnl.gov/data/

In [ ]:
import numpy as np
import pandas as pd
import igraph as ig
import matplotlib.pyplot as plt

from collections import defaultdict

import tqdm

In [ ]:
root = 'dmbd/'

train_labels = root + "truth_labels_train.json"
test_labels = root + 'truth_labels_test.json'
tree_files = ['trees_0.json','trees_1.json','trees_2.json','trees_3.json','trees_4.json','trees_5.json',
              'trees_6.json','trees_7.json']

In [ ]:
# Get Tree Data
tree_file_id = 7
tree_file = root + tree_files[tree_file_id]
df_full = pd.read_json(tree_file)
df_full.head()

In [ ]:
df_tree = df_full[(df_full["RuleName"] == "process_create") | (df_full["RuleName"] == "detonation")]
df_tree

In [ ]:
len(df_tree["experiment"].unique())

In [ ]:
child = set(df_tree.child_guid)
parent = set(df_tree.parent_guid) ## Can be 'None', we'll delete later
## node dictionaries
nodes = parent.union(child)
len(nodes)

In [ ]:
nodes_dict = {v:k for k,v in enumerate(nodes)}
inv_nodes_dict = {k:v for k,v in enumerate(nodes)}

In [ ]:
## build directed graph from edgelist
child = [nodes_dict[x] for x in df_tree.child_guid]
parent = [nodes_dict[x] for x in df_tree.parent_guid]
edges = np.array([parent,child]).T
G = ig.Graph.TupleList(edges, directed=True)
G = G.simplify()
G.vs['guid'] = [inv_nodes_dict[int(x)] for x in G.vs['name']]
G.vcount()

In [ ]:
# Add details
name_dict = dict(zip(df_tree.child_guid, df_tree.child_name))
name_dict.update(dict(zip(df_tree.parent_guid, df_tree.parent_name)))
G.vs["name"] = [name_dict.get(guid, "") for guid in G.vs["guid"]]

parent_dict = dict(zip(df_tree.child_guid, df_tree.parent_guid))
G.vs["parent"] = [parent_dict.get(guid, "") for guid in G.vs["guid"]]

time_dict = dict(zip(df_tree.child_guid, df_tree.timestamp))
G.vs["timestamp"] = [time_dict.get(guid, "") for guid in G.vs["guid"]]

In [ ]:
Trees = G.connected_components(mode="weak")
G.vs['tree'] = Trees.membership
len(Trees) # number of trees

In [ ]:
## drop trees of size < min_tree_size and non-tree(s)
min_tree_size = 3
min_tree_depth = 2

_dct = dict(enumerate(Trees.sizes()))
G.vs['tree_size'] = [_dct[i] for i in G.vs['tree']]
# roots = np.where(np.array(G.degree(mode='in'))==0)[0]
# non_tree = set(np.array(G.vs['tree'])).difference(set(np.array(G.vs['tree'])[roots]))

# Tree depth
depths = np.empty(len(Trees), dtype="int64")
for tree_id in tqdm.trange(len(Trees)):
    tree = Trees.subgraph(tree_id)
    root = np.argmin(np.array(tree.degree(mode='in')))
    depth = max(tree.distances(root)[0]) # Trees are padded with extra start root
    depths[tree_id] = depth
G.vs['tree_depth'] = [depths[i] for i in G.vs['tree']]

G.delete_vertices([v for v in G.vs if (v['tree_size']<min_tree_size or v['tree_depth'] < min_tree_depth)])

In [ ]:
## re-compute 
Trees = G.connected_components(mode="weak")
G.vs['tree'] = Trees.membership
len(Trees) # number of trees

In [ ]:
## ex: showing a size 10 component
idx = np.where(np.array(Trees.sizes())==10)[0][0]
sg = Trees.subgraph(idx)
ly = sg.layout_reingold_tilford()
#ig.plot(sg, 'smalltree_1.png', layout=ly, vertex_color='grey', vertex_label_size=0)
fig, ax = plt.subplots()
ig.plot(sg, layout=ly, target=ax, vertex_color='grey', vertex_label_size=0)


In [ ]:
plt.hist(Trees.sizes())
plt.yscale("log")

In [ ]:
roots = np.where(np.array(G.degree(mode='in'))==0)[0]
depths = np.array(G.vs["tree_depth"])[roots]
plt.hist(depths)
plt.yscale('log')

In [ ]:
# Build Smaller df of node in interesting trees
rows = [[guid, parent, name, timestamp] for guid, name, parent, timestamp in zip(G.vs["guid"], G.vs["name"], G.vs["parent"], G.vs["timestamp"])]

In [ ]:
# Add numerical columns from other data
# image_loads, file_ops, net_connects -> number of each operation for each process
image_loads = defaultdict(int)
file_creates = defaultdict(int)
network_connects = defaultdict(int)

df_ops = df_full[df_full["RuleName"].isin(["image_load", "file_create", "net_connect"])]
for i in tqdm.trange(len(df_ops)):
    df_row = df_ops.iloc[i]
    rule = df_row.RuleName
    if rule == "image_load":
        image_loads[df_row.parent_guid] += 1
    elif rule == "file_create":
        file_creates[df_row.parent_guid] += 1
    elif rule == "net_connect":
        network_connects[df_row.parent_guid] += 1

In [ ]:
for row in rows:
    guid = row[0]
    row.append(image_loads[guid])
    row.append(file_creates[guid])
    row.append(network_connects[guid])

In [ ]:
# Add experiment number and wether it is clean/malware
experiment_dict = dict(zip(df_full.child_guid, df_full.experiment))
experiment_dict.update(dict(zip(df_full.parent_guid, df_full.experiment)))
for row in rows:
    guid = row[0]
    row.append(experiment_dict[guid]) # Should never miss

In [ ]:
# Turn rows into dataframe and save stub
cols = ["guid", "parent", "process_name", "timestamp", "image_loads", "file_ops", "network_connects", "experiment"]
smaller_df = pd.DataFrame(rows, columns=cols)
smaller_df["timestamp"] = pd.to_datetime(smaller_df["timestamp"])
smaller_df

In [ ]:
smaller_df.to_feather(f"clean_trees/trees{tree_file_id}.feather")

In [ ]:
# Code for combining
dfs = [pd.read_feather(f"clean_trees/trees{i}.feather") for i in range(8)]

In [ ]:
dfs[0]

In [ ]:
all_trees = pd.concat(dfs, ignore_index=True)
all_trees

In [ ]:
len(all_trees["experiment"].unique())

In [ ]:
# Get Truth Data
df_truth_train = pd.read_json(train_labels)
df_truth_train['test'] = False
#display(df_truth_train.head())
#print(df_truth_train['label'].value_counts())

df_truth_test = pd.read_json(test_labels)
df_truth_test['test'] = True
#display(df_truth_test.head())
#print(df_truth_test['label'].value_counts())

df_truth = pd.concat([df_truth_train, df_truth_test]).reset_index().drop('index',axis=1)
df_truth

In [ ]:
label_dict = dict(zip(df_truth.experiment, df_truth.label))
test_dict = dict(zip(df_truth.experiment, df_truth.test))

all_trees["malicious"] = [label_dict[eno] == "malicious" for eno in all_trees.experiment]
all_trees["test"] = [test_dict[eno] for eno in all_trees.experiment]

In [ ]:
all_trees

In [ ]:
all_trees.to_feather("clean_trees/all_trees.feather")